In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder \
    .appName("RFM_Pipeline") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

26/04/19 08:14:49 WARN Utils: Your hostname, Jonathans-MacBook-Pro-16.local resolves to a loopback address: 127.0.0.1; using 192.168.100.197 instead (on interface en0)
26/04/19 08:14:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 08:14:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/19 08:14:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
pdf = pd.read_excel("../data/raw/online_retail_II.xlsx", engine="openpyxl")
df = spark.createDataFrame(pdf)

In [4]:
df = df.withColumn("TotalPrice", col("Quantity") * col("Price"))
df.show(5)

26/04/16 13:03:55 WARN TaskSetManager: Stage 0 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|        TotalPrice|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|              83.4|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|              81.0|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|              81.0|
| 489434|    22041|RECORD FRAME 7" S...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|100.80000000000001|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|              30.0|
+-------+---------+-----

In [ ]:
df_clean = (
    df
    # Drop entire row duplicates
    .dropDuplicates()
    
    # Drop rows with NaN in numerical columns
    # Drop rows with zero or non-positive Quantity or Price
    .filter(
        (~isnan("Quantity")) &
        (~isnan("Price")) &
        (~isnan("Customer ID")) &
        (col("Quantity") > 0) &
        (col("Price") > 0)
    )
    
    # Cast types safely
    .withColumn("Customer ID", col("Customer ID").cast("int"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("TotalPrice", col("Quantity") * col("Price"))
)

print(f"Final clean dataset: {df_clean.count()} rows")
print(f"Number of rows removed: {df.count() - df_clean.count()}")
print(f"Number of duplicates removed: {df.count() - df.dropDuplicates().count()}")
print(f"Number of NaN rows removed: {df.count() - df.filter((~isnan('Quantity')) & (~isnan('Price')) & (~isnan('Customer ID'))).count()}")
print(f"Number of zero/negative values removed: {df.count() - df.filter((col('Quantity') > 0) & (col('Price') > 0)).count()}")
print(f"Number of rows with invalid Customer ID removed: {df.count() - df.filter(col('Customer ID').isNotNull()).count()}")
df_clean.show(5)


26/04/19 09:13:12 WARN TaskSetManager: Stage 0 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 09:13:17 WARN TaskSetManager: Stage 6 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


Final clean dataset: 400916 rows


26/04/19 09:13:18 WARN TaskSetManager: Stage 9 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 09:13:20 WARN TaskSetManager: Stage 15 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


Number of rows removed: 124545


26/04/19 09:13:21 WARN TaskSetManager: Stage 18 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 09:13:23 WARN TaskSetManager: Stage 24 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


...

Number of zero/negative values removed: 13895


26/04/19 09:13:26 WARN TaskSetManager: Stage 36 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.
26/04/19 09:13:27 WARN TaskSetManager: Stage 39 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


Number of rows with invalid Customer ID removed: 0


26/04/19 09:13:27 WARN TaskSetManager: Stage 42 contains a task of very large size (2092 KiB). The maximum recommended task size is 1000 KiB.


+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|        TotalPrice|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|      13085|United Kingdom|              83.4|
| 489434|    22041|RECORD FRAME 7" S...|      48|2009-12-01 07:45:00|  2.1|      13085|United Kingdom|100.80000000000001|
| 489434|    22064|PINK DOUGHNUT TRI...|      24|2009-12-01 07:45:00| 1.65|      13085|United Kingdom|39.599999999999994|
| 489434|    21523|FANCY FONT HOME S...|      10|2009-12-01 07:45:00| 5.95|      13085|United Kingdom|              59.5|
| 489435|    22350|           CAT BOWL |      12|2009-12-01 07:46:00| 2.55|      13085|United Kingdom|30.599999999999998|
+-------+---------+-----